# 07 Repository index and reproducibility summary

This notebook summarizes the final structure of the dissertation reproducibility repository, verifies expected notebooks and outputs, and creates final inventory tables for public release.

## Imports and paths

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import yaml

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
ARCHIVE_DIR = PROJECT_ROOT / "archive"
FIGURE_DIR = PROJECT_ROOT / "figures"
CHECKPOINT_DIR = PROJECT_ROOT / "outputs" / "checkpoints"
DOCS_DIR = PROJECT_ROOT / "docs"
CONFIG_PATH = PROJECT_ROOT / "config" / "analysis_config.yml"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)

print("Project root:", PROJECT_ROOT)
print("Notebook directory:", NOTEBOOK_DIR)
print("Processed data:", PROCESSED_DIR)
print("Figures:", FIGURE_DIR)
print("Checkpoints:", CHECKPOINT_DIR)
print("Docs:", DOCS_DIR)
print("Config:", CONFIG_PATH)


## Expected notebook inventory

In [ ]:
expected_notebooks = pd.DataFrame([
    {"notebook": "00_project_setup_and_data_inventory.ipynb", "purpose": "Project setup, path checks, and data inventory", "status": "complete", "run_order": 0},
    {"notebook": "01_recalculate_wo_landscape_from_counts.ipynb", "purpose": "Recalculate pooled frequencies, enrichment metrics, and W/O landscape tables", "status": "complete", "run_order": 1},
    {"notebook": "02_low_read_filter_sensitivity.ipynb", "purpose": "Evaluate sample-level read-depth cutoff sensitivity", "status": "complete", "run_order": 2},
    {"notebook": "03_dissertation_figures_from_landscape_tables.ipynb", "purpose": "Regenerate dissertation figures and summary tables from processed landscape outputs", "status": "complete", "run_order": 3},
    {"notebook": "04_network_and_path_analysis.ipynb", "purpose": "Reproduce network/path analyses and weighted optimal path calculations", "status": "complete", "run_order": 4},
    {"notebook": "05_epistasis_analysis.ipynb", "purpose": "Reproduce pairwise, third-order, and variance-decomposition epistasis analyses", "status": "complete", "run_order": 5},
    {"notebook": "06_chris_model_comparison_and_residual_landscape.ipynb", "purpose": "Reconstruct archived human-model comparison and residual landscape workflow", "status": "complete", "run_order": 6},
    {"notebook": "07_repository_index_and_reproducibility_summary.ipynb", "purpose": "Final repository index and reproducibility summary", "status": "complete", "run_order": 7},
])

expected_notebooks["exists"] = expected_notebooks["notebook"].apply(
    lambda name: (NOTEBOOK_DIR / name).exists()
)

expected_notebooks["relative_path"] = expected_notebooks["notebook"].apply(
    lambda name: str((NOTEBOOK_DIR / name).relative_to(PROJECT_ROOT))
)

expected_notebooks.to_csv(
    CHECKPOINT_DIR / "repository_expected_notebooks.csv",
    index=False
)

expected_notebooks


## Data/input inventory

In [ ]:
top_level_inventory = []

for path in sorted(PROJECT_ROOT.iterdir()):
    if path.name.startswith(".git") or path.name == ".DS_Store":
        continue

    top_level_inventory.append({
        "name": path.name,
        "relative_path": str(path.relative_to(PROJECT_ROOT)),
        "type": "directory" if path.is_dir() else "file",
        "size_bytes": path.stat().st_size if path.is_file() else np.nan,
    })

top_level_inventory = pd.DataFrame(top_level_inventory)

top_level_inventory.to_csv(
    CHECKPOINT_DIR / "repository_top_level_inventory.csv",
    index=False
)

top_level_inventory


In [ ]:
processed_data_inventory = []

for path in sorted(PROCESSED_DIR.glob("*")):
    if path.is_file():
        processed_data_inventory.append({
            "file": path.name,
            "relative_path": str(path.relative_to(PROJECT_ROOT)),
            "size_bytes": path.stat().st_size,
            "suffix": path.suffix,
        })

processed_data_inventory = pd.DataFrame(processed_data_inventory)

processed_data_inventory.to_csv(
    CHECKPOINT_DIR / "repository_processed_data_inventory.csv",
    index=False
)

processed_data_inventory


## Figure inventory

In [ ]:
figure_inventory = []

for path in sorted(FIGURE_DIR.rglob("*")):
    if path.is_file():
        figure_inventory.append({
            "figure_file": path.name,
            "relative_path": str(path.relative_to(PROJECT_ROOT)),
            "figure_group": str(path.parent.relative_to(FIGURE_DIR)),
            "size_bytes": path.stat().st_size,
            "suffix": path.suffix,
        })

figure_inventory = pd.DataFrame(figure_inventory)

figure_inventory.to_csv(
    CHECKPOINT_DIR / "repository_figure_inventory.csv",
    index=False
)

figure_inventory


## Checkpoint/output inventory

In [ ]:
checkpoint_inventory = []

for path in sorted(CHECKPOINT_DIR.glob("*.csv")):
    checkpoint_inventory.append({
        "checkpoint_file": path.name,
        "relative_path": str(path.relative_to(PROJECT_ROOT)),
        "size_bytes": path.stat().st_size,
    })

checkpoint_inventory = pd.DataFrame(checkpoint_inventory)

checkpoint_inventory.to_csv(
    CHECKPOINT_DIR / "repository_checkpoint_inventory.csv",
    index=False
)

checkpoint_inventory


## Reproducibility status table

In [ ]:
reproducibility_status = pd.DataFrame([
    {
        "category": "notebooks",
        "check": "all expected notebooks exist",
        "status": bool(expected_notebooks["exists"].all()),
        "n_items": expected_notebooks.shape[0],
    },
    {
        "category": "processed_data",
        "check": "processed data files inventoried",
        "status": processed_data_inventory.shape[0] > 0,
        "n_items": processed_data_inventory.shape[0],
    },
    {
        "category": "figures",
        "check": "figure files inventoried",
        "status": figure_inventory.shape[0] > 0,
        "n_items": figure_inventory.shape[0],
    },
    {
        "category": "checkpoints",
        "check": "checkpoint csv files inventoried",
        "status": checkpoint_inventory.shape[0] > 0,
        "n_items": checkpoint_inventory.shape[0],
    },
])

reproducibility_status.to_csv(
    CHECKPOINT_DIR / "repository_reproducibility_status.csv",
    index=False
)

reproducibility_status


## Recommended run order

The notebooks are intended to be read and run in numerical order:

1. `00_project_setup_and_data_inventory.ipynb`
2. `01_recalculate_wo_landscape_from_counts.ipynb`
3. `02_low_read_filter_sensitivity.ipynb`
4. `03_dissertation_figures_from_landscape_tables.ipynb`
5. `04_network_and_path_analysis.ipynb`
6. `05_epistasis_analysis.ipynb`
7. `06_chris_model_comparison_and_residual_landscape.ipynb`
8. `07_repository_index_and_reproducibility_summary.ipynb`

Notebook `07` performs no new scientific analysis; it is a repository-level audit and public-release index.

## Final repository summary

This repository reconstructs the dissertation-associated analysis workflow for the SARS-CoV-2 RBD W/O combinatorial genotype landscape.

The workflow recalculates genotype frequencies, enrichment metrics, and landscape tables; evaluates low-read sample sensitivity; regenerates dissertation-facing figures; reconstructs network/path analyses; reproduces pairwise and third-order epistasis analyses; and documents the archived Chris human-model comparison and residual-landscape workflow.

The numbered notebooks are designed to be run in order. Intermediate tables are written to `outputs/checkpoints/`, figures are written to `figures/`, and final inventories are generated in this notebook.

This notebook marks the dissertation reproducibility workflow as complete.